# Table 1 — Robustness of classical deep hedging (Black-Scholes)

Replicates Table 1 of He, Sutter & Gonon (2025):  
**Robustness of classical deep hedging strategy under different attack methods and magnitudes.**

For each CVaR model (α ∈ {0.5, 0.75, 0.95, 0.99}) and each perturbation radius δ,  
we evaluate the ES_α hedging loss on adversarially perturbed GBM paths using:
- **WPGD** — Wasserstein-2 distributional attack (Frank-Wolfe)
- **WBPGD** — Per-path budget / direction decomposition attack

In [1]:
import sys
sys.path.insert(0, '..')

import torch
import numpy as np
import pandas as pd
from pathlib import Path

from src.gbm_simulator import GBMParams, GBMPathGenerator
from src.hedging.hedge_network import HedgeNet
from src.hedging.loss import CVaRLoss
from src.hedging.attacker import DHAttacker

RESULTS_DIR = Path('..') / 'results'
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Running on {device}')

Running on cpu


In [2]:
# Parameters (match training in train_buehler_benchmark.py)
S0, K, sigma, mu = 100.0, 100.0, 0.2, 0.0
N, dt = 30, 1 / 365
T = N * dt

ALPHAS     = [0.5, 0.75, 0.95, 0.99]
DELTA_LIST = [0.0, 0.01, 0.03, 0.05, 0.1, 0.3, 0.5]
RATIO      = 4
N_ITER     = 20

In [3]:
# Generate clean test paths (seed=20 matches buehler_benchmark.ipynb)
params = GBMParams(S0=S0, mu=mu, sigma=sigma, T=T, N=N, M=100_000)
paths  = GBMPathGenerator(params)(seed=20).to(device)
print(f'Test paths: {paths.shape}  on {paths.device}')

Test paths: torch.Size([100000, 31])  on cpu


In [4]:
# Load trained networks + saved z / capital for each alpha
TAGS = {0.5: 'ES05', 0.75: 'ES075', 0.95: 'ES095', 0.99: 'ES099'}

models = {}
for alpha in ALPHAS:
    tag = TAGS[alpha]
    net = HedgeNet(N=N, width=20)
    net.load_state_dict(
        torch.load(RESULTS_DIR / f'buehler_{tag}_network.pt', weights_only=True)
    )
    net = net.to(device).eval()

    log      = torch.load(RESULTS_DIR / f'buehler_{tag}_log.pt', weights_only=False)
    z        = torch.tensor(log['z'],       dtype=torch.float32, device=device)
    capital  = torch.tensor(log['capital'], dtype=torch.float32, device=device)
    models[alpha] = (net, z, capital)
    print(f'  alpha={alpha:4}  z={log["z"]:+.5f}  capital={log["capital"]:.4f}')

  alpha= 0.5  z=-0.01705  capital=2.2871
  alpha=0.75  z=+0.23241  capital=2.2871
  alpha=0.95  z=+0.40181  capital=2.2871
  alpha=0.99  z=+0.42000  capital=2.2871


In [5]:
def eval_es(X: torch.Tensor, alpha: float) -> float:
    """Empirical ES_alpha of hedging errors X, evaluated at optimal (empirical) VaR."""
    z_opt = torch.quantile(X, alpha)
    return (torch.relu(X - z_opt).mean() / (1.0 - alpha) + z_opt).item()

In [6]:
attacker = DHAttacker()
results  = {}   # (alpha, delta) -> {'wpgd': float, 'wbpgd': float}

for alpha in ALPHAS:
    net, z, capital = models[alpha]
    loss_fn = CVaRLoss(K=K, alpha=alpha)
    print(f'\n=== ES_{alpha} ===')

    for delta in DELTA_LIST:
        # WPGD — Wasserstein-2 distributional attack
        _, _, X_wp = attacker.wp_attack_gbm(
            net, paths, loss_fn, capital, z,
            delta=delta, ratio=RATIO, n=N_ITER,
        )

        # WBPGD — per-path budget / direction attack
        _, _, X_bp = attacker.budget_attack_gbm(
            net, paths, loss_fn, capital, z,
            delta=delta, ratio=RATIO, n=N_ITER,
        )

        es_wp = eval_es(X_wp.to(device), alpha)
        es_bp = eval_es(X_bp.to(device), alpha)
        results[(alpha, delta)] = {'wpgd': es_wp, 'wbpgd': es_bp}
        print(f'  delta={delta:.2f}  WPGD={es_wp:.4f}  WBPGD={es_bp:.4f}')


=== ES_0.5 ===
  delta=0.00  WPGD=0.2801  WBPGD=0.0000
  delta=0.01  WPGD=0.3141  WBPGD=0.3242
  delta=0.03  WPGD=0.3846  WBPGD=0.4155
  delta=0.05  WPGD=0.4585  WBPGD=0.5114
  delta=0.10  WPGD=0.6587  WBPGD=0.7714
  delta=0.30  WPGD=1.6988  WBPGD=2.1518
  delta=0.50  WPGD=3.1538  WBPGD=4.0846

=== ES_0.75 ===
  delta=0.00  WPGD=0.4543  WBPGD=0.0000
  delta=0.01  WPGD=0.5047  WBPGD=0.5128
  delta=0.03  WPGD=0.6102  WBPGD=0.6364
  delta=0.05  WPGD=0.7222  WBPGD=0.7688
  delta=0.10  WPGD=1.0307  WBPGD=1.1429
  delta=0.30  WPGD=2.7394  WBPGD=3.2861
  delta=0.50  WPGD=5.2729  WBPGD=6.4294

=== ES_0.95 ===
  delta=0.00  WPGD=0.7785  WBPGD=0.0000
  delta=0.01  WPGD=0.8606  WBPGD=0.8822
  delta=0.03  WPGD=1.0714  WBPGD=1.1681
  delta=0.05  WPGD=1.3267  WBPGD=1.5215
  delta=0.10  WPGD=2.1208  WBPGD=2.6316
  delta=0.30  WPGD=7.4797  WBPGD=10.0736
  delta=0.50  WPGD=16.4682  WBPGD=21.2367

=== ES_0.99 ===
  delta=0.00  WPGD=1.1003  WBPGD=0.0000
  delta=0.01  WPGD=1.2693  WBPGD=1.2503
  delta=0.

In [7]:
# Table 1: rows = delta, columns = (alpha, attack_method)
col_idx = pd.MultiIndex.from_tuples(
    [(f'ES_{a}', m) for a in ALPHAS for m in ['WPGD', 'WBPGD']],
    names=['Model', 'Attack'],
)

rows = []
for delta in DELTA_LIST:
    row = []
    for alpha in ALPHAS:
        row += [results[(alpha, delta)]['wpgd'], results[(alpha, delta)]['wbpgd']]
    rows.append(row)

df = pd.DataFrame(
    rows,
    index=pd.Index(DELTA_LIST, name='δ'),
    columns=col_idx,
)

print('Table 1 — ES_α hedging loss under adversarial attacks (GBM / BS model)\n')
print(df.to_string(float_format='{:.4f}'.format))
df.style.format('{:.4f}')

Table 1 — ES_α hedging loss under adversarial attacks (GBM / BS model)

Model  ES_0.5        ES_0.75        ES_0.95         ES_0.99        
Attack   WPGD  WBPGD    WPGD  WBPGD    WPGD   WBPGD    WPGD   WBPGD
δ                                                                  
0.00   0.2801 0.0000  0.4543 0.0000  0.7785  0.0000  1.1003  0.0000
0.01   0.3141 0.3242  0.5047 0.5128  0.8606  0.8822  1.2693  1.2503
0.03   0.3846 0.4155  0.6102 0.6364  1.0714  1.1681  1.7083  1.7577
0.05   0.4585 0.5114  0.7222 0.7688  1.3267  1.5215  2.2879  2.4803
0.10   0.6587 0.7714  1.0307 1.1429  2.1208  2.6316  4.3119  4.9719
0.30   1.6988 2.1518  2.7394 3.2861  7.4797 10.0736 19.4885 22.2402
0.50   3.1538 4.0846  5.2729 6.4294 16.4682 21.2367 40.6748 45.5316
